# Ensemble Regressors

## Co jsou Ensemble Regressors?

Ensemble regressors (ansámblové regresory) jsou metody strojového učení, které kombinují predikce z více jednotlivých regresních modelů za účelem vytvoření silnějšího a přesnějšího výsledného modelu. Základní myšlenka spočívá v tom, že skupina "slabších" modelů může společně dosáhnout lepšího výkonu než jeden komplexní model.

### Hlavní principy ensemble metod:

1. **Agregace** - Kombinace predikcí z více modelů pro snížení variance a zlepšení stability
2. **Diverzifikace** - Použití různých modelů nebo dat pro zajištění nezávislosti chyb
3. **Váhování** - Možnost přiřadit různým modelům různou důležitost

### Typy ensemble metod pro regresi:

1. **Bagging** (Bootstrap Aggregating) - Trénuje více modelů na různých náhodných podmnožinách trénovacích dat
   - Příklad: Random Forest

2. **Boosting** - Sekvenčně trénuje modely tak, aby se každý nový model zaměřil na chyby předchozích modelů
   - Příklady: Gradient Boosting, AdaBoost, XGBoost

3. **Stacking** - Trénuje nový meta-model, který kombinuje predikce několika základních modelů
   - Příklad: Stacked Regressor

4. **Voting** - Jednoduše průměruje predikce více modelů
   - Příklad: Voting Regressor

V této praktické ukázce se zaměříme na implementaci různých ensemble regresních metod pomocí scikit-learn a porovnáme jejich výkon na různých datových sadách.

In [ ]:
# Import potřebných knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    ExtraTreesRegressor,
    VotingRegressor,
    StackingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing, load_diabetes, make_regression
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Pro reprodukovatelnost výsledků
np.random.seed(42)

# Nastavení vizualizací
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_palette('viridis')

## 1. Random Forest Regressor

Random Forest je metoda založená na baggingu, která kombinuje více rozhodovacích stromů. Každý strom je trénován na náhodně vybrané podmnožině trénovacích dat a příznaků.

### Klíčové vlastnosti:
- **Odolnost vůči přeučení** - Díky průměrování predikcí z více stromů
- **Nízká citlivost na parametry** - Poměrně snadné nastavení
- **Schopnost zachytit nelineární vztahy**
- **Implicitní výběr příznaků**
- **Možnost paralelního zpracování**

In [ ]:
# Načtení datasetu California Housing
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target
feature_names = housing.feature_names

# Rozdělení na trénovací a testovací sady
X_train, X_test, y_train, y_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Základní informace o datasetu
print("Dataset California Housing:")
print(f"Počet vzorků: {X_housing.shape[0]}")
print(f"Počet příznaků: {X_housing.shape[1]}")
print(f"Příznaky: {feature_names}")
print("\nRozdělení dat:")
print(f"Trénovací vzorky: {X_train.shape[0]}")
print(f"Testovací vzorky: {X_test.shape[0]}")

# Základní statistiky cílové proměnné
print("\nStatistiky cílové proměnné (cena domu):")
print(f"Minimum: ${y_housing.min():.2f}")
print(f"Maximum: ${y_housing.max():.2f}")
print(f"Průměr: ${y_housing.mean():.2f}")
print(f"Medián: ${np.median(y_housing):.2f}")
print(f"Směrodatná odchylka: ${y_housing.std():.2f}")

In [ ]:
# Trénování Random Forest Regressoru
rf_regressor = RandomForestRegressor(
    n_estimators=100,  # Počet stromů v lese
    max_depth=None,    # Maximální hloubka stromů (None pro neomezenou)
    min_samples_split=2,  # Minimální počet vzorků pro rozdělení uzlu
    random_state=42
)

# Měření času trénování
start_time = time()
rf_regressor.fit(X_train, y_train)
train_time = time() - start_time

# Predikce
y_pred_rf = rf_regressor.predict(X_test)

# Vyhodnocení modelu
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)

print(f"Random Forest Regressor - výkon:")
print(f"Čas trénování: {train_time:.2f} sekund")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

In [ ]:
# Vizualizace důležitosti příznaků
feature_importance = rf_regressor.feature_importances_
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 8))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(feature_names)[sorted_idx])
plt.title('Důležitost příznaků v Random Forest modelu')
plt.xlabel('Důležitost')
plt.tight_layout()
plt.show()

# Vizualizace skutečných vs. predikovaných hodnot
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Skutečné hodnoty')
plt.ylabel('Predikované hodnoty')
plt.title('Random Forest: Skutečné vs. Predikované hodnoty')
plt.grid(True)
plt.tight_layout()
plt.show()

## 2. Gradient Boosting Regressor

Gradient Boosting je metoda založená na boostingu, která postupně staví na sebe navazující modely tak, že každý nový model se zaměřuje na chyby (rezidua) předchozích modelů.

### Klíčové vlastnosti:
- **Vysoká přesnost** - Často dosahuje nejlepších výsledků mezi neuronovými sítěmi
- **Schopnost zachytit komplexní vztahy** v datech
- **Sekvenční trénování** - Každý model se učí z chyb předchozích modelů
- **Náchylnější k přeučení** než Random Forest - Vyžaduje pečlivější nastavení parametrů
- **Implicitní výběr příznaků**

In [ ]:
# Trénování Gradient Boosting Regressoru
gb_regressor = GradientBoostingRegressor(
    n_estimators=100,  # Počet stromů (boostů)
    learning_rate=0.1,  # Míra učení
    max_depth=3,       # Omezení hloubky stromů pro prevenci přeučení
    subsample=0.8,     # Použití pouze části dat pro každý strom
    random_state=42
)

# Měření času trénování
start_time = time()
gb_regressor.fit(X_train, y_train)
train_time = time() - start_time

# Predikce
y_pred_gb = gb_regressor.predict(X_test)

# Vyhodnocení modelu
mse = mean_squared_error(y_test, y_pred_gb)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_gb)
r2 = r2_score(y_test, y_pred_gb)

print(f"Gradient Boosting Regressor - výkon:")
print(f"Čas trénování: {train_time:.2f} sekund")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

In [ ]:
# Vizualizace křivky učení (redukce chyby s rostoucím počtem stromů)
train_scores = np.zeros((100,), dtype=np.float64)
test_scores = np.zeros((100,), dtype=np.float64)

for i, y_pred in enumerate(gb_regressor.staged_predict(X_train)):
    train_scores[i] = mean_squared_error(y_train, y_pred)
    
for i, y_pred in enumerate(gb_regressor.staged_predict(X_test)):
    test_scores[i] = mean_squared_error(y_test, y_pred)

plt.figure(figsize=(10, 6))
plt.plot(np.arange(100) + 1, train_scores, 'b-', label='Trénovací MSE')
plt.plot(np.arange(100) + 1, test_scores, 'r-', label='Testovací MSE')
plt.xlabel('Počet stromů')
plt.ylabel('Mean Squared Error')
plt.title('Křivka učení Gradient Boosting Regressoru')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Důležitost příznaků
feature_importance = gb_regressor.feature_importances_
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 8))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), np.array(feature_names)[sorted_idx])
plt.title('Důležitost příznaků v Gradient Boosting modelu')
plt.xlabel('Důležitost')
plt.tight_layout()
plt.show()

## 3. Porovnání různých ensemble regresorů

Nyní porovnáme různé typy ensemble regresorů a jejich výkon na našem datasetu.

In [ ]:
# Definice modelů pro porovnání
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostRegressor(DecisionTreeRegressor(max_depth=4), n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42),  # Pro srovnání s jedním modelem
    'Linear Regression': LinearRegression()  # Pro srovnání s jednoduchým modelem
}

# Výsledky pro každý model
results = []

for name, model in models.items():
    # Měření času trénování
    start_time = time()
    model.fit(X_train, y_train)
    train_time = time() - start_time
    
    # Predikce
    y_pred = model.predict(X_test)
    
    # Vyhodnocení modelu
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Uložení výsledků
    results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Čas trénování (s)': train_time
    })

# Převod výsledků na DataFrame pro lepší vizualizaci
results_df = pd.DataFrame(results)
print("Porovnání různých regresních modelů:")
print(results_df.round(4))

In [ ]:
# Vizualizace výsledků porovnání modelů
plt.figure(figsize=(14, 10))

# MSE a R² pro jednotlivé modely
plt.subplot(2, 1, 1)
sns.barplot(x='Model', y='MSE', data=results_df, palette='viridis')
plt.title('Mean Squared Error pro různé regresory')
plt.grid(axis='y')
plt.xticks(rotation=45)

plt.subplot(2, 1, 2)
sns.barplot(x='Model', y='R²', data=results_df, palette='viridis')
plt.title('R² skóre pro různé regresory')
plt.grid(axis='y')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

# Porovnání času trénování
plt.figure(figsize=(12, 6))
sns.barplot(x='Model', y='Čas trénování (s)', data=results_df, palette='coolwarm')
plt.title('Čas trénování jednotlivých modelů')
plt.grid(axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Voting Regressor

Voting Regressor kombinuje více různých regresních modelů a používá jejich průměr (nebo vážený průměr) pro konečnou predikci. Tato metoda může být efektivní, když máte několik dobře fungujících, ale odlišných modelů.

In [ ]:
# Definice základních modelů pro Voting Regressor
base_models = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=100, random_state=42)),
    ('ridge', Ridge(alpha=1.0))  # Přidání jednoduchého lineárního modelu pro diverzitu
]

# Vytvoření Voting Regressoru
voting_regressor = VotingRegressor(estimators=base_models)

# Trénování modelu
start_time = time()
voting_regressor.fit(X_train, y_train)
train_time = time() - start_time

# Predikce
y_pred_voting = voting_regressor.predict(X_test)

# Vyhodnocení modelu
mse = mean_squared_error(y_test, y_pred_voting)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_voting)
r2 = r2_score(y_test, y_pred_voting)

print(f"Voting Regressor - výkon:")
print(f"Čas trénování: {train_time:.2f} sekund")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

# Srovnání s jednotlivými modely v ensemble
print("\nPorovnání Voting Regressoru s jednotlivými modely:")
for name, model in base_models:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    model_r2 = r2_score(y_test, y_pred)
    model_mse = mean_squared_error(y_test, y_pred)
    print(f"{name}: MSE = {model_mse:.4f}, R² = {model_r2:.4f}")

In [ ]:
# Vizualizace predikcí různých modelů včetně Voting Regressoru

# Vytvoření subset dat pro lepší vizualizaci
sample_indices = np.random.choice(len(X_test), size=50, replace=False)
X_sample = X_test[sample_indices]
y_sample = y_test[sample_indices]

# Získání predikcí všech modelů
predictions = {}
predictions['Voting'] = voting_regressor.predict(X_sample)

for name, model in base_models:
    predictions[name] = model.predict(X_sample)

# Vizualizace
plt.figure(figsize=(14, 10))

# Seřazení podle skutečných hodnot pro lepší čitelnost
sort_idx = np.argsort(y_sample)
x_axis = np.arange(len(y_sample))

plt.plot(x_axis, y_sample[sort_idx], 'o-', label='Skutečné hodnoty', linewidth=2)

for name, pred in predictions.items():
    plt.plot(x_axis, pred[sort_idx], 'o-', label=f'{name} predikce', alpha=0.6)

plt.xlabel('Vzorky')
plt.ylabel('Cílová hodnota')
plt.title('Porovnání predikcí různých modelů')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 5. Stacking Regressor

Stacking Regressor je pokročilá ensemble metoda, která kombinuje predikce více "základních" modelů pomocí dalšího "meta" modelu. Na rozdíl od jednoduchého průměrování v Voting Regressoru se Stacking učí, jak optimálně kombinovat základní modely.

In [ ]:
# Definice základních modelů pro Stacking Regressor
base_models_stack = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ('svr', SVR(kernel='linear')),
    ('knn', KNeighborsRegressor(n_neighbors=5))
]

# Definice meta-modelu
meta_model = Ridge(alpha=0.5)

# Vytvoření Stacking Regressoru
stacking_regressor = StackingRegressor(
    estimators=base_models_stack,
    final_estimator=meta_model,
    cv=5  # Počet částí pro cross-validaci při vytváření meta-modelu
)

# Trénování modelu
start_time = time()
stacking_regressor.fit(X_train, y_train)
train_time = time() - start_time

# Predikce
y_pred_stacking = stacking_regressor.predict(X_test)

# Vyhodnocení modelu
mse = mean_squared_error(y_test, y_pred_stacking)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred_stacking)
r2 = r2_score(y_test, y_pred_stacking)

print(f"Stacking Regressor - výkon:")
print(f"Čas trénování: {train_time:.2f} sekund")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R²: {r2:.4f}")

# Srovnání s jednotlivými modely v ensemble
print("\nPorovnání Stacking Regressoru s jednotlivými modely:")
for name, model in base_models_stack:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    model_r2 = r2_score(y_test, y_pred)
    model_mse = mean_squared_error(y_test, y_pred)
    print(f"{name}: MSE = {model_mse:.4f}, R² = {model_r2:.4f}")

In [ ]:
# Srovnání všech ensemble metod
ensemble_models = {
    'Random Forest': models['Random Forest'],
    'Gradient Boosting': models['Gradient Boosting'],
    'AdaBoost': models['AdaBoost'],
    'Extra Trees': models['Extra Trees'],
    'Voting Regressor': voting_regressor,
    'Stacking Regressor': stacking_regressor
}

ensemble_results = []

for name, model in ensemble_models.items():
    # Predikce (použijeme již natrénované modely)
    y_pred = model.predict(X_test)
    
    # Vyhodnocení modelu
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Uložení výsledků
    ensemble_results.append({
        'Model': name,
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })

# Převod výsledků na DataFrame
ensemble_results_df = pd.DataFrame(ensemble_results)
print("Porovnání ensemble metod:")
print(ensemble_results_df.round(4).sort_values('MSE'))

In [ ]:
# Vizualizace porovnání ensemble metod
plt.figure(figsize=(12, 8))

# Seřazení podle výkonu (MSE)
sorted_results = ensemble_results_df.sort_values('MSE')

# Graf MSE
plt.subplot(2, 1, 1)
sns.barplot(x='Model', y='MSE', data=sorted_results, palette='viridis')
plt.title('Porovnání MSE různých ensemble metod')
plt.grid(axis='y')
plt.xticks(rotation=45)

# Graf R²
plt.subplot(2, 1, 2)
sorted_by_r2 = ensemble_results_df.sort_values('R²', ascending=False)
sns.barplot(x='Model', y='R²', data=sorted_by_r2, palette='viridis')
plt.title('Porovnání R² různých ensemble metod')
plt.grid(axis='y')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## 6. Optimalizace hyperparametrů

Pro dosažení nejlepších výsledků je důležité optimalizovat hyperparametry ensemble modelů. Ukážeme si, jak to udělat pro Gradient Boosting Regressor.

In [ ]:
# Definice parametrů pro hledání
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0],
    'min_samples_split': [2, 5]
}

# Pro rychlejší výpočet použijeme pouze část dat
# V reálné aplikaci byste měli použít všechna dostupná data
random_indices = np.random.choice(X_train.shape[0], size=5000, replace=False)
X_train_sample = X_train[random_indices]
y_train_sample = y_train[random_indices]

# Inicializace modelu pro optimalizaci
gb = GradientBoostingRegressor(random_state=42)

# GridSearchCV pro nalezení nejlepších parametrů
grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    cv=3,  # Použijeme menší počet foldů pro rychlejší výpočet
    scoring='neg_mean_squared_error',
    n_jobs=-1,  # Paralelní zpracování
    verbose=1
)

# Zahájení prohledávání
print("Začátek hledání optimálních hyperparametrů...")
start_time = time()
grid_search.fit(X_train_sample, y_train_sample)
search_time = time() - start_time
print(f"Hledání dokončeno za {search_time:.2f} sekund.")

# Výsledky
print(f"\nNejlepší parametry:")
print(grid_search.best_params_)
print(f"Nejlepší skóre (neg_mean_squared_error): {grid_search.best_score_:.4f}")

# Použití nejlepšího modelu na celém datasetu
best_gb = grid_search.best_estimator_
best_gb.fit(X_train, y_train)

# Vyhodnocení optimalizovaného modelu
y_pred_best_gb = best_gb.predict(X_test)
mse_best_gb = mean_squared_error(y_test, y_pred_best_gb)
r2_best_gb = r2_score(y_test, y_pred_best_gb)

print(f"\nVýkon optimalizovaného Gradient Boosting modelu:")
print(f"MSE: {mse_best_gb:.4f}")
print(f"R²: {r2_best_gb:.4f}")

## 7. Ensemble regressors na jiném datasetu: Diabetes

Pro lepší pochopení chování ensemble regresorů, aplikujeme je na jiný dataset - diabetes - který má menší počet vzorků a příznaků.

In [ ]:
# Načtení diabetes datasetu
diabetes = load_diabetes()
X_diabetes = diabetes.data
y_diabetes = diabetes.target

# Základní informace o datasetu
print("Dataset Diabetes:")
print(f"Počet vzorků: {X_diabetes.shape[0]}")
print(f"Počet příznaků: {X_diabetes.shape[1]}")
print(f"Příznaky: {diabetes.feature_names}")

# Rozdělení na trénovací a testovací sady
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=42
)

# Definice modelů pro testování
diabetes_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostRegressor(DecisionTreeRegressor(max_depth=3), n_estimators=100, random_state=42),
    'Voting Regressor': VotingRegressor([
        ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
        ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
        ('ridge', Ridge(alpha=1.0))
    ])
}

# Trénování a vyhodnocení modelů
diabetes_results = []

for name, model in diabetes_models.items():
    # 5-násobná cross-validace pro robustnější vyhodnocení
    cv_scores = cross_val_score(model, X_diabetes, y_diabetes, 
                              cv=5, scoring='r2', n_jobs=-1)
    
    # Trénování na celém trénovacím setu
    model.fit(X_train_d, y_train_d)
    y_pred = model.predict(X_test_d)
    
    # Metriky
    mse = mean_squared_error(y_test_d, y_pred)
    r2 = r2_score(y_test_d, y_pred)
    
    diabetes_results.append({
        'Model': name,
        'MSE': mse,
        'R²': r2,
        'CV R² (střed)': np.mean(cv_scores),
        'CV R² (std)': np.std(cv_scores)
    })

# Výpis výsledků
diabetes_results_df = pd.DataFrame(diabetes_results)
print("\nVýsledky na diabetes datasetu:")
print(diabetes_results_df.round(4).sort_values('MSE'))

In [ ]:
# Vizualizace výsledků na diabetes datasetu
plt.figure(figsize=(14, 10))

# Seřazení podle výkonu
sorted_results_d = diabetes_results_df.sort_values('MSE')

# Graf MSE
plt.subplot(2, 1, 1)
sns.barplot(x='Model', y='MSE', data=sorted_results_d, palette='viridis')
plt.title('MSE různých modelů na diabetes datasetu')
plt.grid(axis='y')
plt.xticks(rotation=45)

# Graf cross-validačních skóre
plt.subplot(2, 1, 2)
plt.errorbar(
    x=range(len(sorted_results_d)),
    y=sorted_results_d['CV R² (střed)'],
    yerr=sorted_results_d['CV R² (std)'],
    fmt='o',
    capsize=5
)
plt.xticks(range(len(sorted_results_d)), sorted_results_d['Model'], rotation=45)
plt.title('Cross-validační R² skóre (s odchylkami)')
plt.ylabel('R²')
plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Vytvoření vlastního ensemble modelu s validační sadou

Ukážeme si, jak vytvořit vlastní ensemble model s použitím validační sady pro určení vah jednotlivých modelů.

In [ ]:
# Rozdělení trénovacích dat na trénovací a validační sadu
X_train_m, X_val, y_train_m, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

# Definice základních modelů
base_models_custom = [
    ('rf', RandomForestRegressor(n_estimators=100, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ('et', ExtraTreesRegressor(n_estimators=100, random_state=42)),
    ('ridge', Ridge(alpha=1.0))
]

# Trénování základních modelů
trained_models = {}
val_predictions = {}
val_scores = {}

for name, model in base_models_custom:
    # Trénování modelu
    model.fit(X_train_m, y_train_m)
    trained_models[name] = model
    
    # Predikce na validační sadě
    y_val_pred = model.predict(X_val)
    val_predictions[name] = y_val_pred
    
    # Výpočet skóre na validační sadě
    val_mse = mean_squared_error(y_val, y_val_pred)
    val_r2 = r2_score(y_val, y_val_pred)
    val_scores[name] = {'MSE': val_mse, 'R²': val_r2}
    
    print(f"Model {name} - Validační MSE: {val_mse:.4f}, R²: {val_r2:.4f}")

# Výpočet vah pro modely na základě inverzní MSE
mse_values = np.array([val_scores[name]['MSE'] for name, _ in base_models_custom])
inv_mse = 1.0 / mse_values
weights = inv_mse / np.sum(inv_mse)

print("\nVypočítané váhy modelů:")
for (name, _), weight in zip([m for m in base_models_custom], weights):
    print(f"{name}: {weight:.4f}")

# Funkce pro vážený průměr predikcí
def weighted_ensemble_predict(X, models, model_names, model_weights):
    predictions = np.zeros(len(X))
    for (name, _), weight in zip(models, model_weights):
        model = trained_models[name]
        predictions += weight * model.predict(X)
    return predictions

# Predikce na testovací sadě
y_pred_custom = weighted_ensemble_predict(
    X_test, base_models_custom, [m[0] for m in base_models_custom], weights
)

# Vyhodnocení výkonu
custom_mse = mean_squared_error(y_test, y_pred_custom)
custom_r2 = r2_score(y_test, y_pred_custom)

print(f"\nVlastní vážený ensemble model - výkon:")
print(f"MSE: {custom_mse:.4f}")
print(f"R²: {custom_r2:.4f}")

In [ ]:
# Srovnání vlastního modelu s ostatními
model_comparisons = {
    'Random Forest': trained_models['rf'].predict(X_test),
    'Gradient Boosting': trained_models['gb'].predict(X_test),
    'Extra Trees': trained_models['et'].predict(X_test),
    'Ridge': trained_models['ridge'].predict(X_test),
    'Vážený Ensemble': y_pred_custom,
    'Voting Regressor': VotingRegressor([(name, model) for name, model in base_models_custom]).fit(X_train_m, y_train_m).predict(X_test)
}

comparison_results = []

for name, predictions in model_comparisons.items():
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    comparison_results.append({
        'Model': name,
        'MSE': mse,
        'R²': r2
    })

comparison_df = pd.DataFrame(comparison_results)
print("Porovnání všech metod:")
print(comparison_df.round(4).sort_values('MSE'))

## 9. Shrnutí a doporučení pro použití ensemble regresorů

Na základě provedených experimentů můžeme shrnout hlavní zjištění a poskytnout doporučení pro použití různých typů ensemble regresorů.

### Hlavní zjištění:

1. **Ensemble metody obecně překonávají jednotlivé modely** - Jak jsme viděli v našich experimentech, ensemble regresory typicky dosahují lepších výsledků než jednotlivé modely jako lineární regrese nebo samostatné rozhodovací stromy.

2. **Gradient Boosting a Random Forest** - Tyto dva modely patří k nejsilnějším samostatným ensemble metodám, přičemž Gradient Boosting často poskytuje nejlepší výkon, ale za cenu větší výpočetní náročnosti a rizika přeučení.

3. **Stacking a váhování** - Kombinování různých typů modelů pomocí stacking nebo váženého průměrování může dále zlepšit výkon, zejména když základní modely zachycují různé aspekty dat.

4. **Významné zvýšení výkonu díky optimalizaci hyperparametrů** - Správné nastavení hyperparametrů (např. počet stromů, learning rate, hloubka stromů) může výrazně zlepšit výkon ensemble modelů.

### Doporučení pro výběr a použití ensemble regresorů:

1. **Random Forest:**
   - **Kdy použít:** Když potřebujete robustní, snadno použitelný model s minimem nastavování hyperparametrů.
   - **Výhody:** Méně náchylný k přeučení, snadno paralelizovatelný, automaticky zpracovává chybějící hodnoty a nevyžádané příznaky.
   - **Nevýhody:** Může být méně přesný než Gradient Boosting pro některé problémy.

2. **Gradient Boosting:**
   - **Kdy použít:** Když je prioritou maximální přesnost a máte čas na ladění hyperparametrů.
   - **Výhody:** Vysoká přesnost, flexibilita, dobrá schopnost modelovat komplexní vztahy.
   - **Nevýhody:** Náchylný k přeučení bez pečlivého nastavení, sekvenční povaha omezuje paralelizaci.

3. **AdaBoost:**
   - **Kdy použít:** Pro jednodušší regresní problémy, zejména s odlehlými hodnotami.
   - **Výhody:** Jednoduchá implementace, méně náchylný k přeučení než Gradient Boosting.
   - **Nevýhody:** Může být méně přesný pro komplexní problémy.

4. **Extra Trees:**
   - **Kdy použít:** Když potřebujete rychlejší alternativu k Random Forest s podobnou přesností.
   - **Výhody:** Rychlejší trénování díky náhodnému rozdělování uzlů namísto hledání optimálních dělení.
   - **Nevýhody:** Může mít mírně nižší přesnost než Random Forest pro některé problémy.

5. **Voting Regressor:**
   - **Kdy použít:** Když máte několik dobře fungujících modelů s podobnou přesností.
   - **Výhody:** Jednoduché využití více modelů, může zlepšit stabilitu predikcí.
   - **Nevýhody:** Prostý průměr nemusí být optimálním způsobem kombinace modelů.

6. **Stacking Regressor:**
   - **Kdy použít:** Když máte několik odlišných modelů a chcete automaticky nalézt optimální způsob jejich kombinace.
   - **Výhody:** Schopnost zachytit různé vzory v datech díky kombinaci modelů, často vyšší přesnost.
   - **Nevýhody:** Složitější implementace, vyšší výpočetní náročnost, riziko přeučení meta-modelu.

### Obecné rady pro práci s ensemble regresory:

1. **Začněte jednoduše** - Random Forest je často dobrou volbou pro začátek díky své robustnosti a malým nárokům na nastavení.

2. **Používejte cross-validaci** - Vždy používejte k-násobnou cross-validaci pro vyhodnocování a porovnání modelů, abyste získali spolehlivější odhad výkonu.

3. **Optimalizujte hyperparametry** - Gradient Boosting a další pokročilé modely mohou výrazně těžit z pečlivého ladění hyperparametrů.

4. **Zpracování příznaků** - I když ensemble metody jsou robustní vůči irelevantním příznakům, správné předzpracování dat může dále zlepšit výkon.

5. **Kombinujte různé typy modelů** - Pro stacking nebo voting vybírejte modely, které se navzájem doplňují (např. kombinace lineárních a nelineárních modelů).

6. **Zvažte výpočetní nároky** - Ensemble metody mohou být výpočetně náročné, zejména při velkých datových sadách. Pokud je rychlost kritickým faktorem, zvažte jednodušší modely nebo omezení velikosti ensemblu.